# 06. 객체 추적(Tracking) — 탐지 응용 (보너스)

탐지는 **프레임마다** 물체를 찾습니다. 추적은 여기에 **ID를 부여해 프레임을 넘어 같은 물체를 이어**줍니다.

```
탐지: 프레임1 [사람, 사람]   프레임2 [사람, 사람]   (누가 누군지 모름)
추적: 프레임1 [사람#1, 사람#2] 프레임2 [사람#1, 사람#2] (같은 사람 = 같은 번호)
```

배우는 것
1. `track()` 기본 사용
2. 추적 ID 다루기
3. 트래커 종류 (ByteTrack vs BoT-SORT)
4. 활용 아이디어 (개수 세기 등)

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import torch
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
model = YOLO("yolo26n.pt")
print("device:", DEVICE)

## 1. 기본 추적

`track()`은 `predict()`와 거의 같지만, 결과에 **추적 ID**(`boxes.id`)가 추가됩니다. 비디오/스트림에서 진가를 발휘하지만, 개념 확인은 이미지로도 됩니다.

> 비디오가 있다면 `source="video.mp4"`로 바꾸고 `save=True`를 주면 ID가 그려진 결과 영상이 저장됩니다.

In [ ]:
# 이미지 한 장으로 track() 동작 확인 (persist=False)
res = model.track(
    "https://ultralytics.com/images/bus.jpg",
    device=DEVICE,
    persist=False,
    verbose=False,
)[0]

b = res.boxes
print("탐지 수:", len(b))
print("추적 ID:", None if b.id is None else b.id.tolist())

## 2. 프레임 시퀀스에서 ID 유지 (persist=True)

실제 추적은 **연속 프레임**을 순서대로 넣으며 `persist=True`로 상태를 이어갑니다. 아래는 같은 이미지를 여러 번 넣어 ID가 유지되는 구조를 보여주는 예시입니다.

```python
import cv2
cap = cv2.VideoCapture("video.mp4")
while cap.isOpened():
    ok, frame = cap.read()
    if not ok:
        break
    res = model.track(frame, persist=True, device=DEVICE, verbose=False)[0]
    # res.boxes.id 로 각 물체의 ID 사용
```

In [ ]:
# persist=True 로 두 번 연속 추적 → 같은 물체가 같은 ID 를 유지하는지
model = YOLO("yolo26n.pt")   # 트래커 상태 초기화
img = "https://ultralytics.com/images/bus.jpg"
for step in range(2):
    r = model.track(img, persist=True, device=DEVICE, verbose=False)[0]
    ids = None if r.boxes.id is None else [int(i) for i in r.boxes.id]
    print(f"step {step}: ids = {ids}")

## 3. 트래커 종류

Ultralytics는 두 가지 내장 트래커를 제공합니다. `tracker=` 인자로 선택.

| 트래커 | 특징 |
|---|---|
| **botsort.yaml** (기본) | 외형 정보까지 사용, 더 정확, 살짝 무거움 |
| **bytetrack.yaml** | 빠르고 가벼움, 실시간에 유리 |

```python
model.track(source, tracker="bytetrack.yaml")
```

In [ ]:
# ByteTrack 으로 추적
model = YOLO("yolo26n.pt")
r = model.track(img, tracker="bytetrack.yaml", persist=True, device=DEVICE, verbose=False)[0]
print("ByteTrack ids:", None if r.boxes.id is None else [int(i) for i in r.boxes.id])

## 4. 활용 아이디어

추적 ID가 있으면 이런 게 가능해집니다:
- **고유 개수 세기**: 지금까지 등장한 서로 다른 ID 수 = 총 몇 명/몇 대가 지나갔나
- **라인 통과 카운팅**: 특정 선을 넘은 ID 집계 (입/출차, 유동인구)
- **체류 시간**: 같은 ID가 화면에 머문 프레임 수
- **경로 그리기**: ID별 중심 좌표를 이어 이동 궤적 표시

```python
seen_ids = set()
# 매 프레임:
#   if r.boxes.id is not None:
#       seen_ids.update(int(i) for i in r.boxes.id)
# 최종: len(seen_ids) == 누적 고유 객체 수
```

---
### 커리큘럼 전체 완료 🎉
00 개요 → 01 추론 → 02 데이터셋 → 03 학습 → 04 검증 → 05 배포 → 06 추적

> 더 파고 싶으면: 세그멘테이션(`-seg`), 포즈(`-pose`), 분류(`-cls`) 노트북을 추가로 만들 수 있어요.